# Tech Job Market Insights

Data preparation and exploratory analysis.

## 1. Setup

In [ ]:
%pip install -q "pandas>=2" pyarrow matplotlib langdetect

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

## 2. Load and inspect

### Load the dataset

In [ ]:
url = "https://huggingface.co/datasets/zalizedata/tech-job-postings-salary-dataset/resolve/main/data/S/jobs-00000-of-00001.parquet"
df = pd.read_parquet(url)
print("Dataset loaded successfully!")
df.head()

### Inspect the dataset

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print(df.columns.tolist())
df.info()
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(
    ascending=False
)
print(missing_values)
df.describe()

## 3. Clean and validate

### Create a working copy

In [ ]:
df_clean = df.copy()
print("Original dataset shape:", df.shape)
print("Working dataset shape:", df_clean.shape)

### Remove duplicate job IDs

In [ ]:
# Keep the latest scrape for each job ID.
duplicate_count = df_clean["job_id"].duplicated().sum()
print("Duplicated job IDs:", duplicate_count)
df_clean = df_clean.sort_values(
    "scraped_at",
    ascending=False
)
df_clean = df_clean.drop_duplicates(
    subset="job_id",
    keep="first"
).copy()
print("Dataset shape after removing duplicates:", df_clean.shape)
print("Remaining duplicated IDs:", df_clean["job_id"].duplicated().sum())

### Rename columns

In [ ]:
df_clean = df_clean.rename(columns={
    "company_name": "company",
    "title": "job_title",
    "salary_min": "min_salary",
    "salary_max": "max_salary",
    "posted_at": "posting_date",
    "skills": "required_skills"
})
print(df_clean.columns.tolist())

### Trim text values

In [ ]:
text_columns = [
    "company",
    "job_title",
    "country",
    "city",
    "department",
    "employment_type",
    "location_raw"
]
for col in text_columns:
    df_clean[col] = df_clean[col].str.strip()
print("Text columns cleaned successfully!")

### Convert salary values

In [ ]:
salary_columns = ["min_salary", "max_salary"]
for col in salary_columns:
    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors="coerce"
    )
print("Salary columns converted successfully!")

### Convert dates

In [ ]:
# Parse mixed timestamp formats and normalize to UTC.
for col in ["posting_date", "scraped_at"]:
    df_clean[col] = pd.to_datetime(
        df_clean[col], format="mixed", errors="coerce", utc=True
    )

display(df_clean[["posting_date", "scraped_at"]].isna().sum())

### Fill missing categories

In [ ]:
categorical_columns = [
    "company_industry", "company_domain", "employment_type",
    "city", "country", "department",
]
for col in categorical_columns:
    df_clean[col] = df_clean[col].fillna("Unknown")

print(df_clean.isnull().sum().sort_values(ascending=False))

### Validate salary ranges

In [ ]:
invalid_salary = df_clean[
    (df_clean["min_salary"] < 0)
    | (df_clean["max_salary"] < 0)
    | (df_clean["min_salary"] > df_clean["max_salary"])
]
print("Invalid salary records:", len(invalid_salary))

salary_data = df_clean.dropna(subset=["min_salary", "max_salary"])
print("Jobs with salary information:", len(salary_data))
display(salary_data[
    ["job_title", "min_salary", "max_salary", "salary_currency", "salary_period"]
].head(10))

## 4. Prepare the dashboard dataset

### Detect title languages

In [ ]:
from langdetect import DetectorFactory, LangDetectException, detect

DetectorFactory.seed = 0


def detect_language(text):
    if pd.isna(text) or not str(text).strip():
        return "unknown"
    try:
        return detect(str(text))
    except LangDetectException:
        return "unknown"


df_clean["title_language"] = df_clean["job_title"].map(detect_language)
display(df_clean["title_language"].value_counts())

non_english_sample = df_clean.loc[
    df_clean["title_language"].ne("en"), ["job_title", "title_language"]
].head(10)
display(non_english_sample)

### Filter titles for the dashboard

In [ ]:
# The character filter can retain non-English titles.
title_mask = df_clean["job_title"].str.fullmatch(
    r"[A-Za-z0-9\s.,/&()'’+:#\-]+", na=False
)
dashboard_jobs = df_clean.loc[title_mask].copy()

print("Original dataset:", df_clean.shape)
print("Filtered dataset:", dashboard_jobs.shape)
print("Excluded records:", len(df_clean) - len(dashboard_jobs))
retained_percentage = len(dashboard_jobs) / len(df_clean) * 100 if len(df_clean) else 0
print(f"Titles retained: {retained_percentage:.2f}%")
display(dashboard_jobs[["job_title", "country"]].head(15))

### Validate and export the dashboard dataset

In [ ]:
print("Rows:", dashboard_jobs.shape[0])
print("Columns:", dashboard_jobs.shape[1])
print("Missing values:", dashboard_jobs.isnull().sum().sum())
print("Duplicate job IDs:", dashboard_jobs["job_id"].duplicated().sum())
print("Earliest posting:", dashboard_jobs["posting_date"].min())
print("Latest posting:", dashboard_jobs["posting_date"].max())

dashboard_jobs.to_csv("tech_jobs_cleaned.csv", index=False)
print("Saved tech_jobs_cleaned.csv.")

## 5. Explore and visualize

### Employment type mode

In [ ]:
employment_mode = df_clean["employment_type"].mode()
print("Most common employment type(s):", employment_mode.tolist())

### Explore the dashboard dataset

In [ ]:
display(dashboard_jobs.describe())

for col in ["job_title", "country", "employment_type", "company"]:
    print(f"Most frequent values in {col}:")
    display(dashboard_jobs[col].value_counts().head(5))

### Compare skills across three roles

In [ ]:
# Count each skill at most once per posting.
titles = ["Data Engineer", "Machine Learning Engineer", "AI Engineer"]
selected_jobs = df_clean.loc[
    df_clean["job_title"].isin(titles),
    ["job_title", "required_skills"]
].reset_index(drop=True)
skills = (
    selected_jobs.explode("required_skills")
    .dropna(subset=["required_skills"])
    .rename_axis("posting_row")
    .reset_index()
    .drop_duplicates(subset=["posting_row", "required_skills"])
)
postings_per_title = (
    skills.groupby("job_title")["posting_row"].nunique()
)
top_skills = skills["required_skills"].value_counts().head(8).index
skill_counts = (
    pd.crosstab(skills["required_skills"], skills["job_title"])
    .reindex(index=top_skills, columns=titles, fill_value=0)
)
skill_pct = skill_counts.div(postings_per_title, axis=1).mul(100)
ax = skill_pct.iloc[::-1].plot.barh(figsize=(10, 6))
ax.set_title("Top 8 Skills Across Three Tech Roles")
ax.set_xlabel("% of postings with recorded skills")
ax.set_ylabel("Skill")
ax.legend(title="Job title")
plt.tight_layout()
plt.show()
print("Postings with recorded skills:")
print(postings_per_title)

### Rank the top 10 job titles

In [ ]:
top_titles = df_clean["job_title"].value_counts().head(10)
ax = top_titles.sort_values().plot.barh(figsize=(9, 5), color="steelblue")
ax.set_title("Top 10 Job Titles by Number of Postings")
ax.set_xlabel("Number of job postings")
ax.set_ylabel("")
ax.bar_label(ax.containers[0], padding=3)
ax.set_xlim(0, top_titles.max() * 1.15)
plt.tight_layout()
plt.show()

## 6. Streamlit dashboard

Run `python -m streamlit run app.py` from the project folder.